In [0]:
from pyspark.sql.functions import *

def scd_type2(source_table,
              target_table,
              business_keys,
              compare_columns):

  

    # Read source & target
    source = spark.table(source_table)
    target = spark.table(target_table)

    # Create hash
    source = source.withColumn(
        "HashValue",
        sha2(concat_ws("||", *compare_columns),256)
    )

    target = target.withColumn(
        "HashValue",
        sha2(concat_ws("||", *compare_columns),256)
    )

    # Build join condition dynamically
    join_condition = [
        source[k] == target[k]
        for k in business_keys
    ]

    # Stage changed/new rows
    stage_df = (
        source.alias("src")
        .join(
            target.filter(col("IsCurrent")==True).alias("tgt"),
            join_condition,
            "left"
        )
        .filter(
            col("tgt.HashValue").isNull()
            |
            (
                col("src.HashValue")
                !=
                col("tgt.HashValue")
            )
        )
    )

    # Exit if nothing changed
    if stage_df.limit(1).count() == 0:
        print("No changes found.")
        return

    stage_df.createOrReplaceTempView("stage_table")

    # Build dynamic MERGE condition
    merge_condition = " AND ".join(
        [f"tgt.{k}=src.{k}" for k in business_keys]
    )

    # Expire old rows
    spark.sql(f"""
        MERGE INTO {target_table} tgt
        USING stage_table src
        ON {merge_condition}
        AND tgt.IsCurrent=true

        WHEN MATCHED THEN
        UPDATE SET
            tgt.EndDate=current_date(),
            tgt.IsCurrent=false,
            tgt.UpdatedTimestamp=current_timestamp()
    """)

    # Insert new versions
    insert_df = (
        stage_df
        .drop("HashValue")
        .withColumn("StartDate", current_date())
        .withColumn("EndDate", lit(None).cast("date"))
        .withColumn("IsCurrent", lit(True))
        .withColumn("CreatedTimestamp", current_timestamp())
        .withColumn("UpdatedTimestamp", current_timestamp())
    )

    insert_df.write.mode("append").saveAsTable(target_table)

    print("SCD Type 2 Load Completed Successfully")